## Denosisng (by run or by group)

With "by_run" ASVs are computed within each run separately, with "by_group" ASVs are computed at the group level.

Group level potentially allows for greater sensitivity on rare ASVs, but if the samples are not highly standardized there is a risk on introducing biases due to the treating of a dishomogeneous error source as homogeneous.

In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("../")

import os
from os.path import join
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

In [2]:
from qiime2_utils import compile_manifest_files, update_runs_absolute_paths,full_preprocessing_pipeline

#### Make manifest files and data structure

In [3]:
base_path = "../DATA/"
preprocessing_type = ["by_group","by_run"][1]

# params
metadata_file_path  = join(base_path,"metadata/merged_metadata.parq")
raw_runs_folder     = join(base_path,"data/raw_reads/")
preprocessing_dir   = join(base_path,"data/preprocessing/denoising")
batch_size          = 5000
additional_groupby_fields = ["instrument_model"]

#%CAREFUL! may delete the whole preprocessing outcomes
full_preprocessing_path_cleaning = False

# run
manifest_file_paths = compile_manifest_files(raw_runs_folder=raw_runs_folder,
                                             preprocessing_dir=preprocessing_dir,
                                             metadata_file_path=metadata_file_path,
                                             additional_groupby_fields=additional_groupby_fields,
                                             batch_size=batch_size,
                                             full_preprocessing_path_cleaning=full_preprocessing_path_cleaning,
                                             preprocessing_type=preprocessing_type)


manifest_file_paths.to_csv(join(base_path,"metadata",f"{preprocessing_type}_manifest_file_paths.tsv"),sep="\t")

#### Preprocess

In [4]:
%norun
base_path = "../DATA/"
preprocessing_type = ["by_group","by_run"][1]

# import manifest file paths
manifest_file_paths = pd.read_csv(join(base_path,"metadata",f"{preprocessing_type}_manifest_file_paths.tsv"),sep="\t").pipe( lambda x : x.set_index(list(x.columns[:x.shape[1]-1])).iloc[:,-1])
update_runs_absolute_paths(manifest_file_paths) # ubdates raw_runs absolute paths in manifest files to match current system

# exec params
parallel = True
n_jobs   = 25

# computation params
cores       = 1 if parallel else 5  # with more than 1 core cutadapt fails when parallel=True
q_threshold = 30
clear_path  = False
from_step   = 0     # initial step of the preprocessing pipeline
to_step     = 3     # final step of the preprocessing pipeline (included)
overwrite   = False
skip_failed = False


# function compiled with assigned parameters
def compiled_function(manifest_file_path):
    import gc
    full_preprocessing_pipeline(manifest_file_path,
                                cores=cores,
                                q_threshold=q_threshold,
                                from_step=from_step,
                                to_step=to_step,
                                clear_path=clear_path,
                                overwrite=overwrite,
                                skip_failed=skip_failed)
    gc.collect()
    return None


# SEQUENTIAL [run over each group (defined by a manifest file)]
if not parallel:
    for manifest_file_path in manifest_file_paths:
        compiled_function(manifest_file_path)

# PARALLEL [run over each group (defined by a manifest file)]
elif parallel:
    from joblib import Parallel, delayed
    Parallel(n_jobs=n_jobs)(delayed(compiled_function)(mfp) for mfp in manifest_file_paths)

UsageError: Line magic function `%norun` not found.


## Compute ASVs at different sensitivities


In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("../")

import os
from os.path import join
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

In [31]:
## step 3 denoise and ASVs extraction, with different omega_a

from qiime2_utils import denoise_ASVs_extraction2
import qiime2

parallel = True
cores    = 1 if parallel else 10
n_jobs   = 30

base_path = "../DATA/"
runsdir = join(base_path,"data/preprocessing/denoising/by_run")
Qiime_dir = [join(runsdir,run) for run in os.listdir(runsdir)]
    

# iterate over different omega_a values (default=1e-40)
#Omega_a = np.logspace(-10,-100,10)
Omega_a = np.logspace(-1,-100,20)
#Omega_a = [1e-1,1e-40,1e-100]


# function run in parallel over multiple threads
def thred_function(qiime_dir):

    print("\n3*) denoise and ASVs extraction (with omega_a control)")
    print(qiime_dir)

    # import previous steps results
    seq = qiime2.Artifact.load(join(qiime_dir,'1_seq_noprimers.qza'))
    with open(join(qiime_dir,"2_truncation_length.txt"),mode="r") as f:
        truncation_length=int(f.readline().split("=")[-1]) # import truncation length computed in previous steps

    if truncation_length>100:
        # ASV outputs containers
        Asv_tab    = []
        Asv_seq    = []
        Asv_stats  = []
        Derep_counts = []
        Derep_map    = []
        

        # compute ASV for different amega_a values
        for omega_a in Omega_a:
            print(f"\n--------------------------------------------------\n{qiime_dir}\n----[{omega_a=}]")
        
            # run
            asv_tab, asv_seq, asv_stats, derep_counts, derep_map = denoise_ASVs_extraction2(seq,truncation_length,cores=cores,create_patch=False,omega_a=omega_a)
            
            # prepare output
            asv_stats = asv_stats.reset_index().set_index(["sample-id","omega_a"])
            
            # store
            Asv_tab   .append(asv_tab  )
            Asv_seq   .append(asv_seq  )
            Asv_stats .append(asv_stats)
            Derep_counts.append(derep_counts)
            Derep_map   .append(derep_map)
        
        
        # concatenate results into pandas format
        ## asv_tab
        Asv_tab = pd.concat(Asv_tab,axis=1,keys=Omega_a).fillna(0)
        Asv_tab.columns = Asv_tab.columns.set_names(names=["omega_a","sample"])
        Asv_tab.index.name = "ASV_id"
        
        ## asv_seq
        Asv_seq = pd.DataFrame(pd.concat(Asv_seq).drop_duplicates())
        Asv_seq.columns.name="ASV_id"
        
        ## asv_stats
        Asv_stats = pd.concat(Asv_stats)

        ## derep_counts /derep_map
        Derep_counts = pd.concat(Derep_counts,axis=1)
        Derep_map    = pd.concat(Derep_map   ,axis=1)
        Derep_counts.columns = pd.Index(Omega_a,name="omega_a")
        Derep_map   .columns = pd.Index(Omega_a,name="omega_a")


        # save on disk
        savepath = join(qiime_dir,"3_asv")
        os.makedirs(savepath,exist_ok=True)
        Asv_tab.T.sparse.to_dense().to_parquet(join(savepath,"asv_tab.parq"     ))
        Asv_seq                    .to_parquet(join(savepath,"asv_seq.parq"     )) 
        Asv_stats                  .to_parquet(join(savepath,"asv_stats.parq"   ))
        Derep_counts.T             .to_parquet(join(savepath,"derep_counts.parq"))
        Derep_map   .T             .to_parquet(join(savepath,"derep_map.parq"   ))


from qiime2_utils import create_patched_r_script
PATCHED_R_SCRIPT_PATH = "./patched_dada2/run_dada_patched.R"

# --- before the parallel call ---
create_patched_r_script(PATCHED_R_SCRIPT_PATH)  # once, in the main process

Patched R script saved to: ./patched_dada2/run_dada_patched.R
OMEGA_A (upper): 21 times in patched script (expected 21)
omega_a (lower): 3 times in patched script (expected 3)
derep occurrences: 25 (expected > 0)


In [30]:
# SEQUENTIAL
if not parallel:
    for qiime_dir in Qiime_dir:
        thred_function(qiime_dir)

# PARALLEL
elif parallel:
    from joblib import Parallel, delayed
    Parallel(n_jobs=n_jobs)(delayed(thred_function)(qiime_dir) for qiime_dir in Qiime_dir)


3*) denoise and ASVs extraction (with omega_a control)
../DATA/data/preprocessing/denoising/by_run/ERR5329047

--------------------------------------------------
../DATA/data/preprocessing/denoising/by_run/ERR5329047
----[omega_a=0.1]
Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: /mnt/data300/projects/2026_ASV_clustering/2_ASV_extraction/patched_dada2/run_dada_patched.R --input_directory /tmp/qiime2/ubuntu/data/edfd279e-9041-4bfc-a67e-66df97b9c047/data --output_path /tmp/tmpsvzchfon/output.tsv.biom --output_track /tmp/tmpsvzchfon/track.tsv --filtered_directory /tmp/tmpsvzchfon --truncation_length 151 --trim_left 0 --max_expected_errors 2.0 --truncation_quality_score 2 --max_length Inf --pooling_method independent --chimera_method consensus --min_parental_fold 1.0 --allow_one_off False --num_

Loading required package: Rcpp
Loading required package: Rcpp



3*) denoise and ASVs extraction (with omega_a control)
../DATA/data/preprocessing/denoising/by_run/ERR2308888

--------------------------------------------------
../DATA/data/preprocessing/denoising/by_run/ERR2308888
----[omega_a=0.1]
Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: /mnt/data300/projects/2026_ASV_clustering/2_ASV_extraction/patched_dada2/run_dada_patched.R --input_directory /tmp/qiime2/ubuntu/data/4f078354-c2e2-4460-8780-90aeab8af2c6/data --output_path /tmp/tmpm5zwn_0f/output.tsv.biom --output_track /tmp/tmpm5zwn_0f/track.tsv --filtered_directory /tmp/tmpm5zwn_0f --truncation_length 151 --trim_left 0 --max_expected_errors 2.0 --truncation_quality_score 2 --max_length Inf --pooling_method independent --chimera_method consensus --min_parental_fold 1.0 --allow_one_off False --num_

Loading required package: Rcpp



3*) denoise and ASVs extraction (with omega_a control)
../DATA/data/preprocessing/denoising/by_run/ERR2033629

--------------------------------------------------
../DATA/data/preprocessing/denoising/by_run/ERR2033629
----[omega_a=0.1]
Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: /mnt/data300/projects/2026_ASV_clustering/2_ASV_extraction/patched_dada2/run_dada_patched.R --input_directory /tmp/qiime2/ubuntu/data/6bc3de6c-f40e-4676-bafe-aa990236979a/data --output_path /tmp/tmp0__f8krb/output.tsv.biom --output_track /tmp/tmp0__f8krb/track.tsv --filtered_directory /tmp/tmp0__f8krb --truncation_length 150 --trim_left 0 --max_expected_errors 2.0 --truncation_quality_score 2 --max_length Inf --pooling_method independent --chimera_method consensus --min_parental_fold 1.0 --allow_one_off False --num_

Loading required package: Rcpp
Loading required package: Rcpp


... (955 outputs left out)

Loading required package: Rcpp


DADA2: 1.30.0 / Rcpp: 1.0.14 / RcppParallel: 5.1.9 
2) Filtering .
3) Learning Error Rates
8047200 total bases in 53648 reads from 1 samples will be used for learning the error rates.
4) Denoise samples 
.
5) Remove chimeras (method = consensus)
6) Report read numbers through the pipeline
7) Write output

--------------------------------------------------
../DATA/data/preprocessing/denoising/by_run/ERR9735042
----[omega_a=1e-100]
Patch already applied, skipping.
Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: /mnt/data300/projects/2026_ASV_clustering/2_ASV_extraction/patched_dada2/run_dada_patched.R --input_directory /tmp/qiime2/ubuntu/data/1a8817d3-7dd2-4472-ac2c-cb4db8e6bb1d/data --output_path /tmp/tmpc268pu2i/output.tsv.biom --output_track /tmp/tmpc268pu2i/track.tsv --filtered_directory /tmp

Loading required package: Rcpp


DADA2: 1.30.0 / Rcpp: 1.0.14 / RcppParallel: 5.1.9 
2) Filtering .
3) Learning Error Rates
8047200 total bases in 53648 reads from 1 samples will be used for learning the error rates.
4) Denoise samples 
.
5) Remove chimeras (method = consensus)
6) Report read numbers through the pipeline
7) Write output


#### Consistency check between standard qiime2 pipeline, and patched pipeline with default omega_a values

In [33]:
for qiime_dir in Qiime_dir:
    sample_id =  qiime_dir.split("/")[-1]

    # load the esult of standard qiime2 pipeline
    import qiime2
    import biom

    artifact_path = join(qiime_dir,"3_asv_tab.qza")
    if os.path.exists(artifact_path):
        asv_tab = qiime2.Artifact.load(artifact_path)
        counts_default = asv_tab.view(biom.Table).to_dataframe().iloc[:,0]
        counts_default


        # load result of patched qiime2 call with standard omega_a values (should me the same)
        idx = pd.IndexSlice

        tab_filepath = join(qiime_dir,"3_asv","asv_tab.parq")
        counts_default_pathced = pd.read_parquet(tab_filepath).loc[idx[1e-40,sample_id],:]
        counts_default_pathced = counts_default_pathced.iloc[(counts_default_pathced!=0).values] # remove zero-counts asv

        # consistency check
        result = (counts_default_pathced.loc[counts_default.index]==counts_default).all()
        print(sample_id,result)
    else:
        print(sample_id,"Missing")

KeyError: (1e-40, 'ERR1250604')